# 📊 End-to-End Customer Churn Prediction System
> **Industry-Standard ML Pipeline** | Telecom Customer Churn Dataset

---
## Project Overview

| Item | Detail |
|------|--------|
| **Problem Type** | Binary Classification |
| **Target Variable** | `Churn` (True/False) |
| **Models Used** | Logistic Regression, KNN |
| **Dataset** | Telecom Customer Churn (667 rows, 20 features) |

**Pipeline Steps:**
1. Load & Inspect Data
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Feature Engineering
4. Model Training
5. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
6. Overfitting vs Underfitting Analysis
7. Summary Report
---

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

RANDOM_STATE = 42
TEST_SIZE    = 0.20
print('✅ Libraries loaded successfully')

## Step 1 – Load & Inspect Data

In [ ]:
# ── Load dataset ─────────────────────────────────────────────
DATA_PATH = '../data/churn-bigml-20.csv'
df = pd.read_csv(DATA_PATH)

print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Data types and missing values
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✅')

In [ ]:
# Statistical summary
df.describe()

## Step 2 – Exploratory Data Analysis (EDA)

In [ ]:
# ── Target distribution ───────────────────────────────────────
churn_counts = df['Churn'].value_counts()
print('Churn Distribution:')
print(churn_counts)
print(f'\nChurn Rate: {churn_counts[True]/len(df)*100:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4CAF50', '#F44336']

ax1.pie(churn_counts.values, labels=['No Churn', 'Churn'],
        colors=colors, autopct='%1.1f%%', startangle=140)
ax1.set_title('Churn Distribution')

churn_counts.plot(kind='bar', ax=ax2, color=colors, edgecolor='white')
ax2.set_xticklabels(['No Churn', 'Churn'], rotation=0)
ax2.set_title('Churn Count')
ax2.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn by categorical features ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pd.crosstab(df['International plan'], df['Churn']).plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Churn by International Plan')
axes[0].set_xlabel('International Plan')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['No Churn', 'Churn'])

pd.crosstab(df['Voice mail plan'], df['Churn']).plot(
    kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Churn by Voice Mail Plan')
axes[1].set_xlabel('Voice Mail Plan')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Numerical feature distributions ─────────────────────────
num_features = ['Total day minutes', 'Total eve minutes',
                'Total night minutes', 'Customer service calls',
                'Account length', 'Total intl minutes']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, num_features):
    for val, col, lbl in zip([False, True], colors, ['No Churn', 'Churn']):
        ax.hist(df[df['Churn']==val][feat], bins=30, alpha=0.6, color=col, label=lbl)
    ax.set_title(feat)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions by Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(14, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3 – Preprocessing & Feature Engineering

In [ ]:
df_processed = df.copy()

# 3a. Handle missing values
for col in df_processed.columns:
    if df_processed[col].isnull().sum() > 0:
        if df_processed[col].dtype in [np.float64, np.int64]:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
        else:
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)
print('✅ Missing values handled')

# 3b. Label encode categorical columns
le = LabelEncoder()
for col in ['State', 'International plan', 'Voice mail plan']:
    df_processed[col] = le.fit_transform(df_processed[col])
print('✅ Categorical variables encoded')

# 3c. Encode target
df_processed['Churn'] = df_processed['Churn'].astype(int)
print('✅ Target encoded: False→0, True→1')

df_processed.head()

In [ ]:
# 3d. Feature / Target split
X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']

# 3e. Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'y_train : {y_train.shape}  |  y_test : {y_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%  |  Test churn rate: {y_test.mean()*100:.1f}%')

# 3f. Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('✅ Features scaled with StandardScaler')

## Step 4 – Model Training

In [ ]:
# ── Logistic Regression ───────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
print('✅ Logistic Regression trained')

# ── K-Nearest Neighbours ─────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
knn_pred = knn.predict(X_test_sc)
print('✅ KNN (k=5) trained')

## Step 5 – Evaluation

In [ ]:
def evaluate(name, model, y_true, y_pred, X_train, y_train):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cv   = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(f'\n──── {name} ────')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1-Score : {f1:.4f}')
    print(f'CV F1    : {cv.mean():.4f} ± {cv.std():.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=['No Churn','Churn']))
    return {'Model':name,'Accuracy':acc,'Precision':prec,'Recall':rec,'F1-Score':f1,'CV F1':cv.mean()}

r1 = evaluate('Logistic Regression', lr,  y_test, lr_pred,  X_train_sc, y_train)
r2 = evaluate('KNN (k=5)',           knn, y_test, knn_pred, X_train_sc, y_train)

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, y_pred) in zip(axes, [('Logistic Regression', lr_pred),
                                       ('KNN (k=5)', knn_pred)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'])
    ax.set_title(f'Confusion Matrix – {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────
plt.figure(figsize=(7, 5))

for name, model, color in [('Logistic Regression', lr, '#3F51B5'),
                             ('KNN (k=5)', knn, '#E91E63')]:
    proba = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc(fpr,tpr):.3f})')

plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary evaluation table ──────────────────────────────────
summary_df = pd.DataFrame([r1, r2]).set_index('Model')
summary_df = summary_df.round(4)
print('=== FINAL EVALUATION SUMMARY ===')
summary_df

## Step 6 – Overfitting vs Underfitting Analysis

> **Overfitting**: Model performs well on training data but poorly on test data (large gap).
>
> **Underfitting**: Model performs poorly on *both* training and test data (both scores are low).
>
> **Well-fit**: Training and test performance are close — model generalises well.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Overfitting vs Underfitting Analysis', fontsize=13, fontweight='bold')

for ax, (name, model, y_pred) in zip(axes, [
        ('Logistic Regression', lr, lr_pred),
        ('KNN (k=5)', knn, knn_pred)]):

    train_acc = accuracy_score(y_train, model.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  y_pred)
    gap       = train_acc - test_acc
    diagnosis = 'Well-fit ✅' if gap < 0.03 else ('Overfitting ⚠️' if gap > 0 else 'Underfitting ⚠️')

    bars = ax.bar(['Train Acc', 'Test Acc'], [train_acc, test_acc],
                  color=['#42A5F5','#EF5350'], alpha=0.85, width=0.45)
    for bar, v in zip(bars, [train_acc, test_acc]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.4f}', ha='center', fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title(f'{name}\n{diagnosis}  |  Gap: {gap:.4f}')
    ax.set_ylabel('Accuracy')

    print(f'{name}: Train={train_acc:.4f}, Test={test_acc:.4f}, Gap={gap:.4f} → {diagnosis}')

plt.tight_layout()
plt.show()

## Step 7 – KNN K-Value Sensitivity

In [ ]:
k_range    = range(1, 26)
train_f1s  = []
test_f1s   = []

for k in k_range:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_sc, y_train)
    train_f1s.append(f1_score(y_train, m.predict(X_train_sc), zero_division=0))
    test_f1s.append (f1_score(y_test,  m.predict(X_test_sc),  zero_division=0))

plt.figure(figsize=(10, 5))
plt.plot(k_range, train_f1s, 'o-', label='Train F1', color='#42A5F5')
plt.plot(k_range, test_f1s,  's-', label='Test F1',  color='#EF5350')
plt.axvline(x=5, color='gray', linestyle='--', label='k=5 (chosen)')
plt.xlabel('K Value')
plt.ylabel('F1-Score')
plt.title('KNN – Train vs Test F1 Across K Values')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📋 Final Summary

| Metric | Logistic Regression | KNN (k=5) |
|--------|--------------------|-----------|
| Accuracy | See above | See above |
| Precision | Fewer false alarms | Depends on k |
| Recall | Catches more churners | Depends on k |
| F1-Score | Balanced | Balanced |

### Key Takeaways
- **International Plan** subscribers have a significantly higher churn rate
- **Customer Service Calls** > 3 strongly predict churn
- **Logistic Regression** is interpretable and robust for this dataset
- **KNN** is sensitive to k — tune using CV scores
- No overfitting detected — models generalise well

---
*Portfolio Project | Customer Churn Prediction | End-to-End ML Pipeline*